In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

In [ ]:
from src.utils.training_utils import get_device_map

# model_key = "meta-llama/Llama-3.2-3B"
# model_key = "meta-llama/Llama-3.1-8B"
# model_key = "meta-llama/Llama-3.1-70B-Instruct"
model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "meta-llama/Llama-3.1-405B-Instruct"

# model_key = "google/gemma-2-9b-it"
# model_key = "google/gemma-3-12b-it"
# model_key = "google/gemma-2-27b-it"

# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# model_key = "allenai/OLMo-2-1124-7B-Instruct"
# model_key = "allenai/OLMo-7B-0424-hf"

# model_key = "Qwen/Qwen2-7B"
# model_key = "Qwen/Qwen2.5-14B-Instruct"
# model_key = "Qwen/Qwen2.5-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "Qwen/Qwen3-1.7B"
# model_key = "Qwen/Qwen3-4B"
# model_key = "Qwen/Qwen3-8B"
# model_key = "Qwen/Qwen3-14B"
# model_key = "Qwen/Qwen3-32B"

# device_map = get_device_map(model_key, 30, n_gpus=8)
# device_map

In [ ]:
from src.models import ModelandTokenizer

# from transformers import BitsAndBytesConfig

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    # device_map=device_map,
    device_map="auto",
    # quantization_config = BitsAndBytesConfig(
    #     # load_in_4bit=True
    #     load_in_8bit=True
    # )
    attn_implementation="eager",
)

In [ ]:
device_map = get_device_map(mt.name, 80, n_gpus=8)

def module_to_device(module_name):
    for key in device_map:
        if module_name.startswith(key):
            return f"cuda:{device_map[key]}"
    return "cpu"

module_to_device(mt.mlp_module_name_format.format(20))

In [ ]:
from src.selection.data import SelectOneTask, SelectOrderTask

#################################################################################
# TASK_CLS = SelectOrderTask
# prompt_template_idx = 1
TASK_CLS = SelectOneTask
prompt_template_idx = 3
N_DISTRACTORS = 5
OPTION_STYLE = "single_line"
#################################################################################

select_task = TASK_CLS.load(
    path=os.path.join(
        env_utils.DEFAULT_DATA_DIR, 
        "selection", 
        # "profession.json"
        # "nationality.json"
        "objects.json"
    )
)

print(select_task)

## Patching the residual states

In [ ]:
import copy
import random
from src.selection.utils import KeyedSet, get_first_token_id, verify_correct_option
from src.selection.data import SelectionSample
from src.functional import predict_next_token
from src.tokens import prepare_input
from src.selection.data import get_counterfactual_samples_within_task

In [ ]:
patch_sample, clean_sample = get_counterfactual_samples_within_task(
    # patch_category="politician",
    # clean_category="actor",
    mt=mt,
    task=select_task,
    patch_category="fruit",
    clean_category="vehicle",
    filter_by_lm_prediction=True,
    prompt_template_idx=prompt_template_idx,
    option_style=OPTION_STYLE,
    distinct_options=True,
    patch_n_distractors=5,
    clean_n_distractors=5
)

# patch_sample.default_option_style = "single_line"
# clean_sample.default_option_style = "numbered"

print(patch_sample.prompt(), ">>", patch_sample.obj)
print(clean_sample.prompt(), ">>", clean_sample.obj)

In [ ]:
clean_tokenized = prepare_input(tokenizer=mt, prompts=clean_sample.prompt())
print(mt.tokenizer.decode(clean_tokenized.input_ids[0], skip_special_tokens=False))

In [ ]:
from src.tokens import prepare_input
from src.selection.utils import get_first_token_id
from src.functional import interpret_logits, PatchSpec
from itertools import product
from src.utils.typing import TokenizerOutput, ArrayLike
from typing import Optional, Union
from src.functional import get_module_nnsight, untuple, get_hs, predict_next_token

def layer_wise_patching(
    mt: ModelandTokenizer,
    patch_sample: SelectionSample,
    clean_sample: SelectionSample,
    token_indices: list[int] = [-3, -2, -1],
):
    patch_tokenized = prepare_input(
        tokenizer=mt.tokenizer, prompts=patch_sample.prompt()
    )
    clean_tokenized = prepare_input(
        tokenizer=mt.tokenizer, prompts=clean_sample.prompt()
    )

    random_idx = random.choice(
        list(
            set(list(range(len(clean_sample.options))))
            - {
                patch_sample.obj_idx,
                clean_sample.obj_idx,
                clean_sample.metadata["track_type_obj_idx"],
            }
        )
    )

    track_tokens = {
        "predicate_target": clean_sample.metadata["track_type_obj_token_id"],
        "clean_ans": get_first_token_id(clean_sample.obj, mt.tokenizer, prefix=" "),
        "patch_ans": get_first_token_id(patch_sample.obj, mt.tokenizer, prefix=" "),
        "patch_position": get_first_token_id(
            clean_sample.options[patch_sample.obj_idx], mt.tokenizer, prefix=" "
        ),
        "random_distractor": get_first_token_id(
            clean_sample.options[random_idx], mt.tokenizer, prefix=" "
        ),
    }

    ret = {"track_tokens": track_tokens}

    logit_location = (mt.lm_head_name, -1)
    patch_locations = list(product(mt.layer_names, token_indices))
    # patch_locations = []
    print(patch_locations)

    patch_hs = get_hs(
        mt=mt,
        input=patch_tokenized,
        locations=patch_locations + [logit_location],
        return_dict=True,
    )
    patch_logits = patch_hs[logit_location]
    patch_pred, patch_track = interpret_logits(
        tokenizer=mt.tokenizer,
        logits=patch_logits,
        interested_tokens=track_tokens.values(),
    )
    logger.debug(f"patch_pred={[str(pred) for pred in patch_pred]}")
    logger.debug(f"patch_track={patch_track}")
    ret["patch_pred"] = patch_pred
    ret["patch_track"] = patch_track

    clean_hs = get_hs(
        mt=mt,
        input=clean_tokenized,
        locations=patch_locations + [logit_location],
        return_dict=True,
    )
    clean_logits = clean_hs[logit_location]
    clean_pred, clean_track = interpret_logits(
        tokenizer=mt.tokenizer,
        logits=clean_logits,
        interested_tokens=track_tokens.values(),
    )
    logger.debug(f"clean_pred={[str(pred) for pred in clean_pred]}")
    logger.debug(f"clean_track={clean_track}")
    ret["clean_pred"] = clean_pred
    ret["clean_track"] = clean_track

    layer_wise_patching_results = {}
    for layer in mt.layer_names:
        patch_spec = []
        for token_idx in token_indices:
            patch_spec.append(
                PatchSpec(
                    location=(layer, token_idx), patch=patch_hs[(layer, token_idx)]
                )
            )

        # int_pred, int_track = predict_next_token(
        #     mt=mt,
        #     inputs=clean_tokenized,
        #     token_of_interest=track_tokens.values(),
        #     patches=patch_spec
        # )
        int_hs = get_hs(
            mt=mt,
            input=clean_tokenized,
            locations=[logit_location],
            patches=patch_spec,
            return_dict=True,
        )
        int_logits = int_hs[logit_location]
        int_pred, int_track = interpret_logits(
            tokenizer=mt.tokenizer,
            logits=int_logits,
            interested_tokens=track_tokens.values(),
        )

        logger.debug(f"Layer {layer}: int_pred={[str(pred) for pred in int_pred]}")
        layer_wise_patching_results[layer] = {
            "int_pred": int_pred,
            "int_track": int_track,
        }

    ret["layer_wise_patching_results"] = layer_wise_patching_results
    return ret


patching_result = layer_wise_patching(
    mt=mt,
    patch_sample=patch_sample,
    clean_sample=clean_sample,
    token_indices=[-3, -2, -1],
)

In [ ]:
from src.functional import free_gpu_cache
free_gpu_cache()
validation_set = []
validation_limit = 64

while len(validation_set) < validation_limit:
    print(f"sample {len(validation_set)+1} / {validation_limit}")
    patch, clean = get_counterfactual_samples_within_task(
        mt=mt,
        task=select_task,
        filter_by_lm_prediction=True,
        prompt_template_idx=prompt_template_idx,
        option_style=OPTION_STYLE,
        distinct_options=True,
        # n_distractors=N_DISTRACTORS,
        patch_n_distractors=N_DISTRACTORS,
        clean_n_distractors=N_DISTRACTORS
    )
    validation_set.append((clean, patch))

len(validation_set)

In [ ]:
results = []
for clean, patch in validation_set:
    result = layer_wise_patching(
        mt=mt,
        patch_sample=patch,
        clean_sample=clean,
        token_indices=[-3, -2, -1]
    )
    results.append(result)

In [ ]:
# results = [patching_result]

scores = {token_type: [] for token_type in results[0]["track_tokens"].keys()}
for result in results:
    clean_track = result["clean_track"]
    patch_track = result["patch_track"]

    for token_type in scores.keys():
        layerwise_scores = []
        token_id = result["track_tokens"][token_type]
        for layer_idx in range(mt.n_layer):
            score = result["layer_wise_patching_results"][mt.layer_names[layer_idx]]["int_track"][token_id][1].logit
            layerwise_scores.append(score)
        scores[token_type].append(layerwise_scores)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

plt.figure(figsize=(12, 6))
for token_type, layerwise_scores_list in scores.items():
    # Compute mean and std deviation across results for each layer
    mean_scores = np.mean(layerwise_scores_list, axis=0)
    sterr_scores = np.std(layerwise_scores_list, axis=0) / np.sqrt(len(layerwise_scores_list))

    plt.plot(mean_scores, label=f"{token_type}")
    plt.fill_between(range(len(mean_scores)), mean_scores - sterr_scores, mean_scores + sterr_scores, alpha=0.2)

plt.xlabel("Layer")
plt.ylabel("Logit(x)")
plt.title(f"Residual | {mt.name.split('/')[-1]}")
plt.legend()
plt.show()

## Loading and calculating the basis directions

In [ ]:
import numpy as np

cached_states_dir = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "selection/cached_states",
    mt.name.split("/")[-1],
    "last_token"
)
sample_file_name = "sample_00170.npz"

sample_states = np.load(
    os.path.join(cached_states_dir, sample_file_name),
    allow_pickle=True,
)
print(sample_states.files)
list(sample_states["states"].item().keys())

In [ ]:
sample = SelectionSample.from_dict(sample_states["sample"].item())
tokenized = TokenizerOutput(data=sample.metadata["tokenized"])
print(torch.Tensor(tokenized.input_ids).shape)
print(mt.tokenizer.decode(tokenized.input_ids[0], skip_special_tokens=False))

In [ ]:
import numpy as np

#######################################################
# LIMIT = 128
LIMIT = len(os.listdir(cached_states_dir))
#######################################################

cached_states = {}

for idx, file_name in enumerate(os.listdir(cached_states_dir)[:LIMIT]):
    sample_states = np.load(
        os.path.join(cached_states_dir, file_name), allow_pickle=True
    )
    states = {}
    for key, value in sample_states["states"].item().items():
        layer_idx, token_idx = key.split("_<>_")
        device = module_to_device(layer_idx)
        states[layer_idx] = torch.Tensor(value).to(mt.dtype).to(device)

    for layer_idx in states:
        if layer_idx not in cached_states:
            cached_states[layer_idx] = []
        cached_states[layer_idx].append(states[layer_idx])

    if (idx + 1) % 1000 == 0:
        logger.info(
            f"Processed {idx+1}/{LIMIT} files... ({(idx+1) / LIMIT * 100:.2f}%)"
        )

cached_states = {
    layer_name: torch.stack(cached_states[layer_name], dim=0)
    .to(mt.dtype)
    .to(module_to_device(layer_name))
    for layer_name in cached_states
}

free_gpu_cache()

for key in cached_states:
    print(f"{key}: {cached_states[key].device}, {cached_states[key].shape}")

In [ ]:
import os
from src.functional import free_gpu_cache
from src.utils.typing import SVD

basis_save_path = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "selection",
    "basis_directions",
    mt.name.split("/")[-1],
    "pca",
    "objects",
    "last_token",
)
os.makedirs(basis_save_path, exist_ok=True)

for layer_idx in cached_states:
    print(layer_idx)
    X = cached_states[layer_idx]
    X_centered = X - X.mean(dim=0, keepdim=True)
    svd = SVD.calculate(X_centered)
    basis_directions = svd.V.to(mt.dtype)
    with open(os.path.join(basis_save_path, f"{layer_idx}.pt"), "wb") as f:
        torch.save(basis_directions, f)

In [ ]:
from src.utils.typing import SVD

basis_save_path = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "selection",
    "basis_directions",
    mt.name.split("/")[-1],
    "svd",
    "objects",
    "last_token",
)
os.makedirs(basis_save_path, exist_ok=True)

for layer_idx in cached_states:
    print(layer_idx)
    svd = SVD.calculate(cached_states[layer_idx])
    basis_directions = svd.V.T.to(mt.dtype)
    with open(os.path.join(basis_save_path, f"{layer_idx}.pt"), "wb") as f:
        torch.save(basis_directions, f)

## Loading the calculated basis directions

In [ ]:
from src.functional import free_gpu_cache
basis_save_path = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "selection",
    "basis_directions",
    mt.name.split("/")[-1],
    "svd",
    "objects",
    "last_token",
)
basis_directions = {}
layer_names = [mt.layer_name_format.format(layer_idx) for layer_idx in range(28, 40)]

for layer_idx in layer_names:
    with open(os.path.join(basis_save_path, f"{layer_idx}.pt"), "rb") as f:
        basis_directions[layer_idx] = torch.load(f)# .to(f"cuda:{torch.cuda.device_count()-1}")
    print(
        layer_idx, basis_directions[layer_idx].shape, basis_directions[layer_idx].device
    )

free_gpu_cache()

## Train Subspace

In [ ]:
from src.functional import get_module_nnsight

def apply_patch_with_projection(
    mt,
    clean_prompts,
    patch_prompts,
    projections,
    token_idx = -1,
):
    
    with mt.trace() as tracer:

        # cache states for patching
        patch_hs = {}
        with tracer.invoke(patch_prompts):
            for layer_name in projections:
                module = get_module_nnsight(mt, layer_name)
                current_states = (
                    module.output
                    if ("mlp" in layer_name or layer_name == mt.embedder_name)
                    else module.output[0]
                )
                if current_states.ndim == 2:
                    current_states = current_states.unsqueeze(0)
                patch_hs[layer_name] = current_states[:, token_idx, :].clone()
        
        # apply the patch
        with tracer.invoke(clean_prompts):
            for layer_name in projections:
                module = get_module_nnsight(mt, layer_name)
                current_states = (
                    module.output
                    if ("mlp" in layer_name or layer_name == mt.embedder_name)
                    else module.output[0]
                )
                if current_states.ndim == 2:
                    current_states = current_states.unsqueeze(0)
                clean_h = current_states[:, token_idx, :].clone()

                # apply the projection
                # print(f"{layer_name} | {patch_hs[layer_name].device=} | {projections[layer_name].device=}")
                device=clean_h.device
                patch_proj = torch.matmul(patch_hs[layer_name].to(device), projections[layer_name].to(device))
                clean_proj = torch.matmul(clean_h.to(device), projections[layer_name].to(device))
                current_states[:, token_idx, :] = clean_h - clean_proj + patch_proj
                # current_states[:, token_idx, :] = patch_hs[layer_name]
            
            # get the logits after the intervention
            logits = mt.lm_head.output[:, -1].save()

        # del patch_hs

    return logits

In [ ]:
masks = {
    layer_name: torch.ones(
        mt.n_embd, 
        dtype=mt.dtype, 
        device=module_to_device(layer_name),
        # device=f"cuda:{torch.cuda.device_count()-1}",
        requires_grad=True
    )
    for layer_name in basis_directions.keys()
}

dummy_projections = {}
for layer_idx in basis_directions.keys():
    mask = masks[layer_idx]
    basis_direction = basis_directions[layer_idx]
    masked_directions = basis_direction * mask[:, None]
    dummy_projections[layer_idx] = masked_directions.t() @ masked_directions

In [ ]:
from src.functional import free_gpu_cache
from src.selection.data import get_counterfactual_samples_within_task

free_gpu_cache()
train_set = []
train_limit = 512
# train_limit=64

while len(train_set) < train_limit:
    print(f"sample {len(train_set)+1} / {train_limit}")
    patch, clean = get_counterfactual_samples_within_task(
        mt=mt,
        task=select_task,
        filter_by_lm_prediction=True,
        prompt_template_idx=prompt_template_idx,
        option_style=OPTION_STYLE,
        distinct_options=True,
        # n_distractors=N_DISTRACTORS,
        patch_n_distractors=N_DISTRACTORS,
        clean_n_distractors=N_DISTRACTORS
    )
    train_set.append((clean, patch))

len(train_set)

In [ ]:
from typing import Literal
from src.selection.utils import get_first_token_id

batch_size = 8

target_obj: Literal["predicate_target", "patch_position"] = "predicate_target"

targets = []
clean_samples = []
patch_samples = []

for clean_sample, patch_sample in train_set[: batch_size]:
    objs = {
        "predicate_target": clean_sample.metadata["track_type_obj_token_id"],
        "patch_position": get_first_token_id(
            clean_sample.options[patch_sample.obj_idx], mt.tokenizer, prefix=" "
        )
    }
    print(patch_sample.prompt())
    print(clean_sample.prompt())
    print(f"{objs[target_obj]}: {mt.tokenizer.decode(objs[target_obj])}")

    print("-" * 50)

    targets.append(objs[target_obj])
    patch_samples.append(patch_sample)
    clean_samples.append(clean_sample)

In [ ]:
from src.tokens import prepare_input
from src.utils.typing import TokenizerOutput

prompts = []
prompts.extend([sample.prompt() for sample in clean_samples])
prompts.extend([sample.prompt() for sample in patch_samples])
tokenized = prepare_input(
    prompts=prompts, tokenizer=mt
)

clean_tokenized = TokenizerOutput(
    data={k: v[: len(clean_samples), :] for k, v in tokenized.items()}
)
patch_tokenized = TokenizerOutput(
    data={k: v[len(clean_samples) :, :] for k, v in tokenized.items()}
)

clean_tokenized.input_ids.shape, patch_tokenized.input_ids.shape

In [ ]:
from src.functional import interpret_logits

logits = apply_patch_with_projection(
    mt=mt,
    clean_prompts=clean_tokenized,
    patch_prompts=patch_tokenized,
    projections=dummy_projections,
    token_idx=-1,
)   

mt._model.zero_grad()
free_gpu_cache()

print(f"{logits.shape=}")

for logit in logits:
    pred = interpret_logits(tokenizer=mt, logits = logit)
    print([f"{str(p)}" for p in pred])

target_logits = [logit[tok] for logit, tok in zip(logits, targets)]
print(target_logits)
torch.stack(target_logits).mean()

In [ ]:
from torch.optim import Adam
from src.selection.data import SelectionSample


def get_optimal_projection(
    mt: ModelandTokenizer,
    train_set: list[tuple[SelectionSample, SelectionSample]],
    basis_directions: dict[str, torch.Tensor],
    target: Literal["predicate_target", "patch_position"] = "predicate_target",
    learning_rate: float = 1e-2,
    n_epochs: int = 5,
    lamb=1e-5,
    batch_size: int = 8,
):
    masks = {
        layer_name: torch.full(
            (mt.n_embd,),
            0.5,
            dtype=mt.dtype,
            device=module_to_device(layer_name),
            # device=f"cuda:{torch.cuda.device_count()-1}",
            requires_grad=True,
        )
        for layer_name in basis_directions.keys()
    }
    optimizer = Adam(masks.values(), lr=learning_rate)

    batches = []
    for batch_start in range(0, len(train_set), batch_size):
        batches.append(train_set[batch_start : batch_start + batch_size])

    losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0
        for batch_idx, batch in enumerate(batches):
            clean_samples, patch_samples = zip(*batch)

            prompts = []
            prompts.extend([sample.prompt() for sample in clean_samples])
            prompts.extend([sample.prompt() for sample in patch_samples])
            tokenized = prepare_input(prompts=prompts, tokenizer=mt)

            clean_tokenized = TokenizerOutput(
                data={k: v[: len(clean_samples), :] for k, v in tokenized.items()}
            )
            patch_tokenized = TokenizerOutput(
                data={k: v[len(clean_samples) :, :] for k, v in tokenized.items()}
            )
            batch_targets = []
            batch_distractors = []
            if target == "predicate_target":
                batch_targets = [
                    clean_sample.metadata["track_type_obj_token_id"]
                    for clean_sample in clean_samples
                ]
                batch_distractors = [
                    [
                        get_first_token_id(tokenizer=mt.tokenizer, name=opt, prefix=" ")
                        for idx, opt in enumerate(clean_sample.options)
                        if idx != clean_sample.metadata["track_type_obj_idx"]
                    ]
                    for clean_sample in clean_samples
                ]
            elif target == "patch_position":
                batch_targets = [
                    get_first_token_id(
                        clean_sample.options[patch_sample.obj_idx],
                        tokenizer=mt.tokenizer,
                        prefix=" ",
                    )
                    for clean_sample, patch_sample in zip(clean_samples, patch_samples)
                ]
                batch_distractors = [
                    [
                        get_first_token_id(tokenizer=mt.tokenizer, name=opt, prefix=" ")
                        for idx, opt in enumerate(clean_sample.options)
                        if idx != patch_sample.obj_idx
                    ]
                    for clean_sample, patch_sample in zip(clean_samples, patch_samples)
                ]
            # make sure that information about the patch options isn't being carried
            patch_options = [
                [
                    get_first_token_id(tokenizer=mt.tokenizer, name=opt, prefix=" ")
                    for opt in patch_sample.options
                ]
                for patch_sample in patch_samples
            ]

            # debugging
            # print(f"{len(clean_samples)=}, {len(patch_samples)=}")
            # print(f"{len(batch_targets)=}, {len(batch_distractors)=}")
            # print(f"{len(patch_options)=}")
            # for idx in range(len(clean_samples)):
            #     print(patch_samples[idx].prompt(), ">>", patch_samples[idx].obj)
            #     print(clean_samples[idx].prompt(), ">>", clean_samples[idx].obj)
            #     print(
            #         f'target: {batch_targets[idx]}  ["{mt.tokenizer.decode(batch_targets[idx])}"]'
            #     )
            #     print(f"distractors={[mt.tokenizer.decode(tok) for tok in batch_distractors[idx]]}")
            #     print(f"patch_options={[mt.tokenizer.decode(tok) for tok in patch_options[idx]]}")
            #     print("-" * 50)

            projections = {}
            for layer_name in basis_directions.keys():
                mask = masks[layer_name]
                basis_direction = basis_directions[layer_name]
                # print(f"{layer_name} | {mask.device} | {basis_direction.device}")
                masked_directions = basis_direction * mask[:, None]
                # V directions are row-wise
                projections[layer_name] = masked_directions.t() @ masked_directions

            logits = apply_patch_with_projection(
                mt=mt,
                clean_prompts=clean_tokenized,
                patch_prompts=patch_tokenized,
                projections=projections,
                token_idx=-1,
            )

            # calculate target loss
            target_logits = [logit[tok] for logit, tok in zip(logits, batch_targets)]
            target_loss = -torch.stack(target_logits).mean()  # need this to go up

            # calculate distractor loss
            distractor_logits = [
                logit[distractor_tokens].mean()
                for logit, distractor_tokens in zip(logits, batch_distractors)
            ]
            distractor_loss = 0.1 * torch.stack(distractor_logits).mean()

            # patch option loss
            patch_option_logits = [
                logit[patch_option_tokens].mean()
                for logit, patch_option_tokens in zip(logits, patch_options)
            ]
            patch_option_loss = 0.1* torch.stack(patch_option_logits).mean()

            # mask loss
            mask_l1_loss = None
            for mask in masks.values():
                if mask_l1_loss is None:
                    mask_l1_loss = lamb * mask.norm(p=1)
                else:
                    mask_l1_loss += lamb * mask.norm(p=1).to(mask_l1_loss.device)

            loss = (
                target_loss
                + distractor_loss
                + patch_option_loss
                + mask_l1_loss.to(target_loss.device)
            )
            logger.debug(
                f"Epoch={epoch+1} | {batch_idx=} |>> {target_loss.item():.4f} + {distractor_loss.item():.4f} + {patch_option_loss.item():.4f} + {mask_l1_loss.item():.4f} = {loss.item():.4f}"
            )

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # clamp the masks to [0, 1] after optimization step
            with torch.no_grad():
                for mask in masks.values():
                    # mask.clamp_(0, 1)
                    # mask += 1e-4  # to avoid zero gradients
                    mask.data = torch.sigmoid(mask.data * 5 - 2.5)  # Steeper sigmoid

            epoch_loss += loss.item()
            losses.append(loss.item())
            del (
                projections,
                logits,
            )
            free_gpu_cache()

        num_batches = (
            len(clean_samples) + batch_size - 1
        ) // batch_size  # ceiling division
        logger.debug(f"Epoch {epoch + 1}/{n_epochs}, Loss: {epoch_loss / num_batches}")
        mt._model.zero_grad()
        free_gpu_cache()

    # build projections
    final_projections = {}

    for layer_name in basis_directions.keys():
        mask = masks[layer_name].clamp(0, 1).round().detach()
        basis_direction = basis_directions[layer_name]
        masked_directions = basis_direction * mask[:, None]
        # V directions are row-wise
        final_projections[layer_name] = masked_directions.t() @ masked_directions
        masks[layer_name] = mask.cpu()

    metadata = {
        "losses": losses,
        "masks": masks,
    }

    return final_projections, metadata

In [ ]:
mt._model.zero_grad()
free_gpu_cache()

In [ ]:
projections, metadata = get_optimal_projection(
    mt=mt,
    train_set=train_set,
    basis_directions=basis_directions,
    lamb=1e-3,
    learning_rate=1e-2,
    batch_size=8,
    n_epochs=5,
)

In [ ]:
from matplotlib import pyplot as plt 
plt.plot(metadata["losses"])
plt.show()

In [ ]:
import numpy as np
npz_file = "save_test_subspace.npz"

with open(npz_file, "wb") as f:
    np.savez_compressed(
        f,
        losses=metadata["losses"],
        masks={
            layer_name: mask.cpu().to(torch.float32).detach().numpy()
            for layer_name, mask in metadata["masks"].items()
        },
        allow_pickle=True,
    )

In [ ]:
subspace_optimization_results = np.load(npz_file, allow_pickle=True)
plt.plot(subspace_optimization_results["losses"])
plt.show()

In [ ]:
from src.functional import free_gpu_cache
from src.selection.data import get_counterfactual_samples_within_task

free_gpu_cache()
validation_set = []
validation_limit = 256
# validation_limit=64

while len(validation_set) < validation_limit:
    print(f"sample {len(validation_set)+1} / {validation_limit}")
    patch, clean = get_counterfactual_samples_within_task(
        mt=mt,
        task=select_task,
        filter_by_lm_prediction=True,
        prompt_template_idx=prompt_template_idx,
        option_style=OPTION_STYLE,
        distinct_options=True,
        # n_distractors=N_DISTRACTORS,
        patch_n_distractors=N_DISTRACTORS,
        clean_n_distractors=N_DISTRACTORS
    )
    validation_set.append((clean, patch))
len(validation_set)

In [ ]:
from src.functional import predict_next_token

clean_sample, patch_sample = validation_set[40]

print(patch_sample.prompt(), ">>", patch_sample.obj)
print(clean_sample.prompt(), ">>", clean_sample.obj)

clean_tokenized = prepare_input(prompts=[clean_sample.prompt()], tokenizer=mt)
patch_tokenized = prepare_input(prompts=[patch_sample.prompt()], tokenizer=mt)

track_tokens = {
    "clean_obj": get_first_token_id(
        clean_sample.obj, tokenizer=mt.tokenizer, prefix=" "
    ),
    "patch_obj": get_first_token_id(
        patch_sample.obj, tokenizer=mt.tokenizer, prefix=" "
    ),
    "predicate_target": clean_sample.metadata["track_type_obj_token_id"],
    "patch_position": get_first_token_id(
        clean_sample.options[patch_sample.obj_idx], tokenizer=mt.tokenizer, prefix=" "
    ),
}

interested_tokens = list(
    set(
        list(track_tokens.values())
        + [
            get_first_token_id(opt, tokenizer=mt.tokenizer, prefix=" ")
            for opt in clean_sample.options
        ]
    )
)
clean_pred, clean_track = predict_next_token(
    mt=mt, inputs=clean_tokenized, token_of_interest=interested_tokens
)
logger.info(f"clean_pred={[str(pred) for pred in clean_pred]}")
logger.info(f"{clean_track=}")

In [ ]:
proj_logits = apply_patch_with_projection(
    mt=mt,
    clean_prompts=clean_tokenized,
    patch_prompts=patch_tokenized,
    projections=projections,
    token_idx=-1
)

proj_pred, proj_track = interpret_logits(
    tokenizer=mt.tokenizer,
    logits=proj_logits,
    interested_tokens=interested_tokens
)
logger.info(f"proj_pred={[str(pred) for pred in proj_pred]}")
logger.info(f"{proj_track=}")

In [ ]:
mt.tokenizer.decode(track_tokens["predicate_target"])